# Fine-tuning no Google Colab

Notebook-base para executar o treino a partir de `resources/finetuning_qa.jsonl` enviado manualmente ao Google Drive.

Fluxo:
1. Instalar dependências
2. Montar o Google Drive
3. Carregar o dataset JSONL
4. Configurar o modelo base
5. Executar o fine-tuning leve com QLoRA
6. Fazer merge do adapter no modelo base e salvar o modelo completo
7. Testar prompts após o treino


In [ ]:
!pip -q install --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --upgrade transformers datasets accelerate peft trl bitsandbytes sentencepiece huggingface_hub


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Defina o Secret HF_TOKEN no Colab'
login(token=HF_TOKEN)

DATASET_PATH = Path('/content/drive/MyDrive/teach-chalenge3/finetuning_qa.jsonl')
OUTPUT_DIR = Path('/content/drive/MyDrive/teach-chalenge3/medqa-finetuned-model')
BASE_MODEL = 'meta-llama/Llama-3.2-1B-Instruct'
MAX_LENGTH = 512

assert DATASET_PATH.exists(), f'Dataset não encontrado em {DATASET_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(DATASET_PATH)
print(OUTPUT_DIR)


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = MAX_LENGTH
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(model.__class__.__name__)
print(tokenizer.__class__.__name__)


In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=str(DATASET_PATH), split='train')
dataset = dataset.train_test_split(test_size=0.05, seed=42)
dataset


In [ ]:
def preprocess_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    tokenized['length'] = [len(ids) for ids in tokenized['input_ids']]
    return tokenized

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=['source'])
tokenized_dataset = tokenized_dataset.filter(lambda example: example['length'] > 0)
tokenized_dataset


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    max_seq_length=max_seq_length,
)


In [ ]:
from pathlib import Path
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

SAVE_STEPS = 100

use_bf16 = is_bfloat16_supported()
last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=25,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=1,
    eval_strategy='epoch',
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    processing_class=tokenizer,
)


In [ ]:
trainer.train(resume_from_checkpoint=last_checkpoint)
model.save_pretrained_merged(str(OUTPUT_DIR), tokenizer, save_method='merged_16bit')


In [ ]:
from transformers import pipeline

generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device_map='auto')
prompt = 'ANSWER THE QUESTION.\n[|Question|] What is the role of antibiotics in bacterial infections?[|eQuestion|]\n\n[|Answer|]'
print(generator(prompt, max_new_tokens=128, do_sample=False, repetition_penalty=1.05)[0]['generated_text'])
